In [8]:
'''
Import and load data set
'''

import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

X_train=pd.read_csv("../output/x_train.csv")
X_test=pd.read_csv("../output/x_test.csv")
y_train=pd.read_csv("../output/y_train.csv").squeeze()
y_test=pd.read_csv("../output/y_test.csv").squeeze()


In [9]:
'''
Create the Decision Tree model, train, and predict
'''

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)


In [ ]:
'''
Predict accuracy and f1 scores
'''

accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred, average="weighted")

# Recall and Precision scores
precision = precision_score(y_test,y_pred,average="weighted", zero_division=0)

recall = recall_score(y_test,y_pred,average="weighted", zero_division=0)

#ROC-AUC
y_pred_proba = dt_model.predict_proba(X_test)

roc_auc = roc_auc_score(y_test,y_pred_proba,multi_class="ovr",average="weighted")


In [ ]:
'''
Print model results
'''

print("Decision Tree Results")
print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted F1-score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")


In [ ]:
'''
Analyze most important features and print
'''

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": dt_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nFeature Importance:")
print(feature_importance)


In [ ]:
'''
Make graph of importance
'''

top_features = feature_importance.head(5)

plt.figure(figsize=(8,5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 5 Most Important Features")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
'''
ROC-AUC Graph
'''

classes = np.unique(y_test)

y_test_bin = label_binarize(y_test, classes=classes)

plt.figure(figsize=(8, 6))

for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_pred_proba[:, i]
    )

    roc_auc_class = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"Class {classes[i]} (AUC = {roc_auc_class:.2f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Decision Tree")
plt.legend()
plt.tight_layout()
plt.show()